# Illustrative Example: Synthetic Spatial PCA Demo

This notebook uses a synthetic geophysical grid to show the same Spatial PCA workflow used by the real case-study runs. The goal is to keep the example inspectable while still calling the repository SPCA, ranking, mapping, and validation functions directly.

![Simple Spatial PCA workflow](../docs/figures/spatial_pca_simple_workflow.png)

**Spatial PCA similarity workflow.** The method compares a known deposit geometry with candidate areas by first sampling the geophysical grid(s) into same-size windows. This ensures that every window is evaluated at the same spatial scale as the reference deposit. The window matrix X is standardized and decomposed with spatial PCA to represent each window by its main spatial patterns. The deposit scores are used to select and weight the PCs that best describe the reference geometry. Each window is then ranked by its deposit-weighted distance d_i, producing a similarity map. The top-ranked prediction windows are finally evaluated against other known testing deposits using footprint recovery and hit metrics.


## Setup

This cell resolves the repository root, imports the shared SPCA functions, and imports the illustrative helper module. The helper module contains only notebook-specific glue such as toy plotting, window-ID bookkeeping, GeoDataFrame assembly, and summary-table construction.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / ".matplotlib-cache"))
sys.path.insert(0, str(repo_root / "src"))

import matplotlib.pyplot as plt
from IPython.display import Image, display
import numpy as np
import pandas as pd
from rasterio.transform import Affine

from spatial_pca.spca.pca import fit_spca
from spatial_pca.spca.ranking import rank_spca_windows
import spatial_pca.examples.illustrative as illustrative
import spatial_pca.validation.diagnostics as diagnostics
import spatial_pca.validation.footprint_recovery as footprint_recovery

illustrative = importlib.reload(illustrative)
diagnostics = importlib.reload(diagnostics)
footprint_recovery = importlib.reload(footprint_recovery)

plot_deposit_scores_and_weights = diagnostics.plot_deposit_scores_and_weights
plot_loading_maps = diagnostics.plot_loading_maps
plot_score_pairs = diagnostics.plot_score_pairs
plot_cumulative_recovery = footprint_recovery.plot_cumulative_recovery
plot_top_windows_overlay = footprint_recovery.plot_top_windows_overlay
validate_footprint_recovery = footprint_recovery.validate_footprint_recovery

plt.rcParams["figure.dpi"] = 120


## Input Data

The synthetic grid is pinned in the repository so the illustrative example is reproducible. Known deposits are represented by their sliding-window IDs throughout the notebook, figures, and output tables.


In [ ]:
data_dir = repo_root / "data" / "Illustrative Example Input Data"
output_dir = repo_root / "outputs" / "Illustrative_Example"
output_dir.mkdir(parents=True, exist_ok=True)

field = np.loadtxt(data_dir / "synthetic_field.csv", delimiter=",", skiprows=1)
metadata = json.loads((data_dir / "metadata.json").read_text())
known_windows = pd.read_csv(data_dir / "known_deposit_windows.csv")
known_windows["window_index"] = known_windows["window_index"].astype(int)
known_windows["label"] = known_windows["window_index"].astype(str)
known_deposit_indices = known_windows["window_index"].tolist()

variable_name = metadata["variable_name"]
win_h = int(metadata["window_height"])
win_w = int(metadata["window_width"])
stride_y = int(metadata["stride_y"])
stride_x = int(metadata["stride_x"])
training_window_index = int(metadata["training_window_index"])

gp_min = float(np.min(field))
gp_max = float(np.max(field))
N = field.shape[0]

print(f"Min: {gp_min:.3f}, Max: {gp_max:.3f}")
known_windows


## Sliding Windows

This cell converts the raster into the same window matrix used by the SPCA pipeline. The first figure shows how the geophysical grid becomes the matrix `X`; the second shows the training and testing deposit windows before any PCA is fit.


In [ ]:
context = illustrative.prepare_univariate_window_context(
    field=field,
    window_shape=(win_h, win_w),
    stride_y=stride_y,
    stride_x=stride_x,
    training_window_index=training_window_index,
    variable_name=variable_name,
)
window_matrix = context.window_matrix
X = context.X
pca_input = context.pca_input
deposit_index = context.deposit_index

input_plot_path = illustrative.plot_grid_matrix_schematic(
    context=context,
    field=field,
    window_shape=window_matrix.window_shape,
    training_window_index=training_window_index,
    output_path=output_dir / "toy_example_grid_x.png",
    gp_min=gp_min,
    gp_max=gp_max,
)
display(Image(filename=str(input_plot_path)))

deposit_context_path = illustrative.plot_known_deposit_windows(
    field=field,
    n_cols=context.n_cols,
    window_shape=window_matrix.window_shape,
    training_window_index=training_window_index,
    known_deposit_indices=known_deposit_indices,
    output_path=output_dir / "toy_example_known_deposit_windows.png",
    variable_name=variable_name,
    gp_min=gp_min,
    gp_max=gp_max,
)
display(Image(filename=str(deposit_context_path)))


## Fit SPCA And Rank Windows

Now the notebook follows the case-study workflow: fit SPCA to the window matrix, compute the deposit-specific weighted distance, remove the appended training-template row, and keep the top-ranked candidate windows.


In [ ]:
pca_result = fit_spca(
    pca_input,
    var_name=variable_name,
    patch_size=window_matrix.window_shape,
)

ranking = rank_spca_windows(
    scores=pca_result.scores,
    eigvals=pca_result.eigvals,
    deposit_index=deposit_index,
    k_pcs=None,
)

n_windows = window_matrix.window_indices_for_mapping.shape[0]
valid_rank_mask = (
    (ranking.ranked_idx != deposit_index)
    & (ranking.ranked_idx < n_windows)
)
top_indices = ranking.ranked_idx[valid_rank_mask]
top_distances = ranking.ranked_dists[valid_rank_mask]
ranked_prediction_indices = top_indices
ranked_prediction_distances = top_distances

top_n = 8
top_table = pd.DataFrame({
    "rank": np.arange(1, top_n + 1),
    "window_index": ranked_prediction_indices[:top_n],
    "distance": ranked_prediction_distances[:top_n],
})
top_table.to_csv(output_dir / f"illustrative_top_{top_n}_predicted_windows.csv", index=False)
top_table


## SPCA Diagnostics

These diagnostics use the same plotting functions as the real univariate runs. The score/weight plot shows which PCs matter most for the training deposit, and the loading maps show the spatial patterns associated with the top-weighted PCs.


In [ ]:
scores_plot_path = plot_deposit_scores_and_weights(
    scores=pca_result.scores,
    explained_variance_ratio=pca_result.explained_variance_ratio,
    weights=ranking.weights,
    deposit_index=deposit_index,
    k_used=ranking.k_used,
    output_path=output_dir / "deposit_scores_and_weights.png",
)
display(Image(filename=str(scores_plot_path)))


In [ ]:
loading_maps_path = plot_loading_maps(
    loadings=pca_result.loadings,
    scores=pca_result.scores,
    weights=ranking.weights,
    deposit_index=deposit_index,
    window_shape=window_matrix.window_shape,
    output_path=output_dir / "loading_maps.png",
    max_pcs=min(4, int(ranking.k_used)),
    feature_mask=window_matrix.feature_mask,
    image_cmap="magma",
    origin="lower",
)
display(Image(filename=str(loading_maps_path)))


## PCA Score Pairs

This diagnostic shows the SPCA score space used for ranking. By default, the panels step through adjacent PCs in descending order of deposit-weight percentage. The open blue circles mark the top-ranked prediction windows in this score space. To force exact pairs, set `score_pair_pc_pairs` to 1-based PC tuples such as `[(1, 3), (4, 2)]`.


In [ ]:
n_score_pair_subplots = 3
score_pair_top_n_to_plot = 3
score_pair_pc_pairs = None  # Example: [(1, 3), (4, 2)]
score_pairs_path = plot_score_pairs(
    comparison_space=ranking.comparison_space,
    weights=ranking.weights,
    deposit_index=deposit_index,
    output_path=output_dir / "score_pairs.png",
    n_pairs=n_score_pair_subplots,
    pc_pairs=score_pair_pc_pairs,
    ranked_idx=top_indices,
    top_n_to_plot=score_pair_top_n_to_plot,
)
if score_pairs_path is not None:
    display(Image(filename=str(score_pairs_path)))
else:
    print("Score-pair plots require at least two PCs.")


## Map Ranked Windows And Validate Recovery

The same validation formulas used for the real cases are applied here. The top-window map displays the five highest-ranked prediction windows, while recovery is computed on the top eight stored in `top_windows_gdf`. The final panel puts the map and recovery curve side by side.


In [ ]:
min_cover = 0.5
transform = Affine.translation(0, 0) * Affine.scale(1, 1)
extent = (0, N, 0, N)

top_windows_gdf, deposits_gdf = illustrative.build_illustrative_geodataframes(
    window_matrix=window_matrix,
    top_indices=top_indices,
    top_distances=top_distances,
    top_n=top_n,
    training_window_index=training_window_index,
    known_windows=known_windows,
    transform=transform,
    crs="EPSG:3857",
)

n_plotting_prediction_windows = 5
top_windows_plot_path = plot_top_windows_overlay(
    top_windows_gdf=top_windows_gdf.head(n_plotting_prediction_windows),
    deposits_gdf=deposits_gdf,
    reference_deposit_index=0,
    background_layers={
        variable_name: {
            "array": field,
            "extent": extent,
            "vmin": gp_min,
            "vmax": gp_max,
        }
    },
    transform=transform,
    output_path=output_dir / f"{variable_name}_Top_{n_plotting_prediction_windows}_Predicted_Windows.png",
    title=f"{variable_name}: Top {n_plotting_prediction_windows} Prediction Windows",
    image_cmap="magma",
    image_origin="lower",
    annotate_indices=True,
    annotation_fontsize=18,
    testing_linewidth=2.2,
    training_linewidth=2.8,
    predicted_linewidth=5.4,
)

recovery = validate_footprint_recovery(
    top_windows_gdf=top_windows_gdf,
    deposits_gdf=deposits_gdf,
    reference_deposit_index=0,
    min_cover=min_cover,
)
hit_label_by_deposit = deposits_gdf["window_id"].astype(int).astype(str).to_dict()
recovery_plot_path = plot_cumulative_recovery(
    recovery,
    output_dir / "cum_curve_unique_dep_hits.png",
    deposit_1based=None,
    min_cover=min_cover,
    title=(
        "Cumulative footprint recovery\n"
        f"Orange = any overlap event, Red = first >= {int(100 * min_cover)}% hit "
        "(labels = test window IDs)"
    ),
    hit_label_by_deposit=hit_label_by_deposit,
)

hit_table = illustrative.build_recovery_hit_table(
    recovery=recovery,
    hit_label_by_deposit=hit_label_by_deposit,
)
hit_table.to_csv(output_dir / "illustrative_known_deposit_recovery.csv", index=False)

summary_plot_path = illustrative.plot_image_pair(
    left_path=top_windows_plot_path,
    right_path=recovery_plot_path,
    output_path=output_dir / "top_windows_and_recovery.png",
)
display(Image(filename=str(summary_plot_path)))

hit_table


## Outputs

After running the notebook, check `outputs/Illustrative_Example/` for:

- `toy_example_grid_x.png`
- `toy_example_known_deposit_windows.png`
- `deposit_scores_and_weights.png`
- `loading_maps.png`
- `score_pairs.png`: SPCA score-pair diagnostic colored by distance to the training deposit, with top-ranked windows highlighted.
- `Synthetic_GP_Top_5_Predicted_Windows.png`
- `cum_curve_unique_dep_hits.png`
- `top_windows_and_recovery.png`
- `illustrative_top_8_predicted_windows.csv`
- `illustrative_known_deposit_recovery.csv`
